# Latent Diffusion & Stable Diffusion Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` stacks a toy 1-D "VAE" (identity encoder + decoder, for demonstration; a real VAE would be a conv net) on top of the DDPM from Lesson 06 and adds class conditioning with classifier-free guidance. It shows that the same diffusion loss works whether you run on raw 1-D values or on encoded values — the key insight.

### Step 1: encoder/decoder

In [ ]:
```python

def encode(x):    return x * 0.5          # toy "compression" to smaller scale

def decode(z):    return z * 2.0

In [ ]:
```

A real VAE has trained weights. For pedagogy, this linear map is enough to show that diffusion operates on `z` without caring about the original data space.

### Step 2: diffusion in `z`-space

Same DDPM as Lesson 06. The data the net sees is `z = E(x)`. After sampling `z_0`, decode with `D(z_0)`.

### Step 3: classifier-free guidance

During training, drop the class label 10% of the time (replace with a null token). At inference, compute both `ε_cond` and `ε_uncond`, then:

In [ ]:
```python

eps_cfg = (1 + w) * eps_cond - w * eps_uncond

In [ ]:
```

`w = 0` = no guidance (full diversity), `w = 3` = default, `w = 7+` = saturated / over-sharp.

### Step 4: text conditioning (concept, not code)

Replace the class label with a frozen text encoder output. Feed the text embedding to the U-Net via cross-attention:

In [ ]:
```python

h = h + CrossAttention(Q=h, K=text_embed, V=text_embed)

In [ ]:
```

This is the only substantive difference between a class-conditional diffusion model and Stable Diffusion.

## Exercises

In [ ]:
1. **Easy.** Run `code/main.py` with guidance `w ∈ {0, 1, 3, 7, 15}`. Record mean sample by class. At what `w` do the class means diverge past the real data means?
2. **Medium.** Swap the toy linear encoder for a tanh-MLP encoder/decoder pair with a reconstruction loss. Retrain diffusion on the new latents. Does sample quality change?
3. **Hard.** Set up a real Stable Diffusion inference with diffusers: load `sdxl-base`, run 30 Euler steps with CFG=7, time it. Now switch to `sdxl-turbo` with 4 steps and CFG=0. Same subject, different quality — describe what changed and why.